# demo_colony

> Phase 18 (`specs/multiagent/phase18/`): colony rules for large Claude teams with faulty members. Part 0's numpy forecast (committed first), the tracing check against Phase 15, the pilot, Stage 1 (N = 8 and 16), the go/no-go verdict, Stage 2 (N = 32 and 64, only on `go` and a raised cap) and cached reruns (`COLONY_PLAN`, `COLONY2_PLAN`).

Artifacts (gitignored): `results/multiagent/colony/{quick,pilot,stage1,stage2}/` (`runs.csv`, `turns.jsonl`, `timing.jsonl`, `metrics.json`, `report.md`, `colony.png`) and `results/multiagent/colony/forecast/` (Part 0). The response cache and spend ledger are Phase 10's (`results/multiagent/llm/`), so one API cap covers all live spend. Rows carry Phase 18's columns (`pooledBound`, `honestPooled`, the trace and goal drift) from `colony_trace.runColonyCell`.

- **Quick** (normal tests): Stage 1's 22 cells at one seed (N 8 and 16), 3 turns, with the fake client; identical at 1 and 8 processes.
- **Tracing check** (`--flags llm`, $0: Phase 14's cached replies): the outcome-based contamination against Phase 15's on Phase 14's N = 8 cells (AC-6).
- **Pilot**, **Stage 1**, **Stage 2** (`--cell-timeout 0 --flags llm`, with `EXPLRL_LLM_LIVE=1`): live, each behind its gate; `EXPLRL_COLONY_STAGE` (`pilot`, the default; `stage1`; `stage2`) sets how far a live run goes, so the pilot never runs straight into Stage 1.

In [ ]:
import json
import os
import re
import shutil
import subprocess
import time

import numpy as np

from explRL.bench_grid import readCSV, writeCSV
from explRL.colony_grid import (COLONY2_PLAN, COLONY_PLAN, colonyPilotGrid, costRule1, costRule2, goNoGo, stage1Quick)
from explRL.colony_report import colonySummaries, figColony, writeColony
from explRL.colony_trace import runColonyCell
from explRL.colony32 import COLONY32_PLAN, stage32Grid, stage32Summaries, writeStage32
from explRL.colony3 import (COLONY3_PLAN, complianceGate, costRule3, reusedRows, runStage3Cell, stage3Grid, stage3PilotGrid,
                            stage3Summaries, writeStage3)
from explRL.liar_trace import cacheOnlyClient
from explRL.colony64 import COLONY64_PLAN, pilotCheck64, stage64Grid, stage64PilotGrid, stage64Summaries, writeStage64
from explRL.colony4 import STAGE4_PLAN, pilotCheck4, ref8Tokens, reusedCells4, stage4Grid, stage4PilotGrid
from explRL.colony4_report import stage4Summaries, writeStage4
from explRL.colony_mech import replayCheck, reusedRows4, runStage4Cell
from explRL.colony_grid import COLONY_SEEDS, stage1Grid
from explRL.coop_grid import coopHash
from explRL.llm_budget import PRICES, Budget
from explRL.llm_grid import HAIKU
from explRL.llm_sched import mergeTiming, runWorkload, timingSummary
from explRL.train import demo_artifact_dir

COLONY = 'multiagent/colony'
WORKERS = min(8, os.cpu_count() or 1)
LLM_WORKERS = 32
CAP = float(COLONY2_PLAN['cap'] or 150.0)            # the user raises it in COLONY2_PLAN before Stage 2
LLM = demo_artifact_dir(None, 'multiagent/llm')
LIVE = os.environ.get('EXPLRL_LLM_LIVE') == '1'
STAGE = os.environ.get('EXPLRL_COLONY_STAGE', 'pilot')    # what a live run may do: pilot, stage1, stage2, stage32, stage3pilot, stage3, stage64pilot, stage64, stage4pilot, stage4b1, stage4b2, stage4
LIVE_SPEC = dict(kind='live', cache=str(LLM / 'cache'), cap=CAP, ledger=str(LLM / 'spend.jsonl'))
FORECAST = LLM.parents[2] / 'specs/multiagent/phase18/colony_predictions.json'     # Part 0 (gate for the pilot)
FORECAST4 = LLM.parents[2] / STAGE4_PLAN['forecast']['file']                        # Stage 4 (gate for its pilot)


def gitCommit():
    try:
        return subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip() or 'unknown'
    except OSError:
        return 'unknown'


def readLog(p):
    return [json.loads(x) for x in open(p)] if p.exists() else []


def runColony(mode, grids, spec, plan=COLONY_PLAN, workers=None, prior=(), analyse=None, run=runColonyCell):
    """Cells through runWorkload (runColonyCell): runs.csv, turns.jsonl, timing.jsonl, report.md, colony.png, metrics.json;
    the analysis reads `prior` rows too (Stage 2: Stage 1's, for the tests and the trend over 8-64); `analyse(rows, prior)` ->
    (summary, write(path, S, meta)) replaces the Stage 1 analysis (the 32-agent deviation)."""
    live = spec.get('kind') == 'live'
    out = demo_artifact_dir(mode, COLONY)
    logs = out / 'turns'
    shutil.rmtree(logs, ignore_errors=True)
    t0 = time.time()
    res = runWorkload(grids, spec, workers or (LLM_WORKERS if live else WORKERS), str(logs), run=run)
    wall = time.time() - t0
    rows = res['rows']
    writeCSV(rows, out / 'runs.csv')
    with open(out / 'turns.jsonl', 'w') as f:
        for p in sorted(logs.glob('*.jsonl')) if logs.exists() else []:
            f.write(p.read_text())
    res['timing'] = timingSummary(mergeTiming(logs, out / 'timing.jsonl'), wall)
    shutil.rmtree(logs, ignore_errors=True)
    if analyse is None:
        S, write = colonySummaries(list(prior) + rows, plan, nboot=plan['nboot'] if live else 200), writeColony
    else:
        S, write = analyse(rows, prior)
    meta = dict(mode=mode, planHash=coopHash(plan), commit=gitCommit(), model=HAIKU, cap=float(spec.get('cap', CAP)), wall=wall,
                pricesChecked=PRICES[HAIKU]['checked'], cost=float(sum(r['cost'] for r in rows)),
                calls=int(sum(r['calls'] for r in rows)), spent=Budget(spec.get('cap', CAP), spec['ledger']).spent() if spec.get('ledger') else 0.0)
    write(out / 'report.md', S, meta)
    if analyse is None:
        figColony(out, S)
    (out / 'metrics.json').write_text(json.dumps(dict(meta, done=res['done'], refused=res['refused'], stopped=res['stopped'],
                                                      timing=res['timing'], summary=S), indent=2, sort_keys=True, default=str) + '\n')
    if live:
        for f in ('metrics.json', 'timing.jsonl'):
            if not (out / f.replace('.', '_live.')).exists():
                shutil.copy(out / f, out / f.replace('.', '_live.'))
    print(f"{mode}: {len(rows)} rows, {meta['calls']} calls, ${meta['cost']:.2f} for these runs, ${meta['spent']:.2f} spent in all, "
          f"{wall:.0f} s; stopped: {res['stopped']}")
    return rows, S, meta


def pilotCheck(rows):
    "Invalid rate, tokens and cost per call, and the plain board's honest loss (clean - liars) at the pilot seed."
    calls = sum(r['calls'] for r in rows) or 1
    v = {r['level']: r for r in rows}
    return dict(invalid=sum(r['invalid'] for r in rows) / calls, costPerCall=sum(r['cost'] for r in rows) / calls,
                inputPerCall=sum(r['inputTokens'] + r['cacheReadTokens'] + r['cacheWriteTokens'] for r in rows) / calls,
                outputPerCall=sum(r['outputTokens'] for r in rows) / calls,
                plainLoss=v['plain|clean']['welfareHonest'] - v['plain|independent']['welfareHonest'])

## Quick: Stage 1's cells with the fake client (pipeline check)

In [ ]:
t0 = time.time()
qcells = stage1Quick()
rows, S, _ = runColony('quick', [('colony1', qcells)], dict(kind='fake'))
assert time.time() - t0 < 300 and len(rows) == len(qcells) == 22
q = demo_artifact_dir('quick', COLONY)
for f in ('runs.csv', 'turns.jsonl', 'timing.jsonl', 'metrics.json', 'report.md', 'colony.png'):
    assert (q / f).stat().st_size > 0, f
read = lambda: tuple((q / f).read_bytes() for f in ('runs.csv', 'turns.jsonl'))
ref = read()
runColony('quick', [('colony1', qcells)], dict(kind='fake', pooled=False), workers=1)
assert read() == ref
assert all(np.isfinite(r['honestPooled']) and 'freeRiding' in r and 'contamination' in r for r in rows)
recs = readLog(q / 'turns.jsonl')
arm = lambda x: x['key'].split('|')[2]
assert all('(agent ' not in x['obs'] for x in recs if arm(x) in ('colony', 'colonyGuards'))       # the tally, not the plain list
assert {(r['level'], r['N'], r['nLiar']) for r in rows if r['level'] == 'colonyGuards|independent'} == {('colonyGuards|independent', 8, 2), ('colonyGuards|independent', 16, 4)}
for p in q.iterdir():
    if p.suffix in ('.csv', '.jsonl', '.json', '.md'):
        assert not re.search(r'sk-ant-|x-api-key|authorization|api[_-]?key', p.read_text(), re.I), p
print(open(q / 'report.md').read()[:1200])

## Part 0 (numpy): the committed forecast

`tools/colony_forecast.py run` (10,080 numpy cells); its predictions must be committed before the pilot.

In [ ]:
if FORECAST.exists():
    P = json.loads(FORECAST.read_text())
    for k, v in P['predictions'].items():
        print(f"{k}: {v['sign']} ({v['estimate']:.3f} [{v['ci'][0]:.3f}, {v['ci'][1]:.3f}])")
else:
    print('no forecast yet: python tools/colony_forecast.py run $PWD')

## Tracing check (`--flags llm`, $0): Phase 18's contamination against Phase 15's (AC-6)

Phase 14's baseline cells with liars at N 8, 16 and 32, replayed from the cache: the outcome-based contamination (`colony_trace`: answers built on a lie) against Phase 15's exposure-based one (`liar_report.mechanics`). AC-6 (revised after measuring): never above Phase 15's in any cell, and rising with N as Phase 15's does; the gap is exposure without uptake.

In [ ]:
#| llm
from explRL.liar_report import mechanics
from explRL.liar_trace import cacheOnlyClient, traceRun
from explRL.colony_trace import replay, traceBoard
from explRL.llm_runs import LLMConfig
from explRL.scale_grid import scaleGrid

if not (LLM / 'cache').exists():
    print('skipped: no response cache')
else:
    means = {}
    for N in (8, 16, 32):
        pairs15 = []
        for c in [c for c in scaleGrid() if c['N'] == N and c['level'] == 'baseline|liar']:
            cfg = LLMConfig.fromDict(c['cfg'])
            tr = traceRun(cfg, c['seed'], cacheOnlyClient(cfg, LLM / 'cache'))
            rp = replay(cfg, c['seed'], cacheOnlyClient(cfg, LLM / 'cache'))
            assert rp.rows == tr.rows                                   # the replay is the run
            pairs15.append((mechanics(tr.trace, tr.board, tr.nLiar)['contaminated'], traceBoard(rp.board, rp.answers, rp.nLiar)['contamination']))
        a, b = np.array(pairs15).T
        assert (b <= a + 1e-12).all(), N
        means[N] = (a.mean(), b.mean())
        print(f'N {N}: Phase 15 {a.mean():.3f}, Phase 18 {b.mean():.3f}, gap per cell {np.round(a - b, 3)}')
    assert means[32][1] > means[8][1] and means[32][0] > means[8][0]   # the same trend

## Pilot (`--flags llm`, live, about 2 200 calls, about $1)

N = 16, `plain` and `colony`, clean and with independent liars, seed 1890 (not analysed). Gates: the forecast committed and `COLONY_PLAN` frozen. AC-8: invalid replies under 5%, cost per call, the plain board's honest loss (stop and ask if none), the cost rule.

In [ ]:
#| llm
if not LIVE:
    print('skipped: set EXPLRL_LLM_LIVE=1 to make live API calls')
else:
    assert FORECAST.exists(), 'commit the Part 0 forecast before the pilot'
    assert COLONY_PLAN['frozen'], 'freeze and commit COLONY_PLAN before the pilot'
    prows, _, _ = runColony('pilot', [('colonyPilot', colonyPilotGrid())], LIVE_SPEC)
    pc = pilotCheck(prows)
    rule = costRule1(pc['costPerCall'])
    print(f"invalid {pc['invalid']:.3f} (band < {COLONY_PLAN['pilot']['bands']['invalid']}); tokens per call {pc['inputPerCall']:.0f} in, "
          f"{pc['outputPerCall']:.0f} out; cost per call ${pc['costPerCall']:.5f}; plain board's honest loss {pc['plainLoss']:.3f}")
    print(f"projected Stage 1: ${rule['projected']:.2f}; cost rule: {rule['verdict']}")

## Stage 1 (`--cell-timeout 0 --flags llm`, live, 176 cells, about $25)

The cells the pilot's cost rule allows; then the summaries, the go/no-go verdict, and the forecast's score (`tools/colony_forecast.py score`).

In [ ]:
#| llm
def stage1Cells():
    "The cells the cost rule allows from the pilot (None: stop and ask)."
    pr = demo_artifact_dir('pilot', COLONY) / 'runs.csv'
    if not pr.exists():
        return None
    pc = pilotCheck(readCSV(pr))
    if not pc['invalid'] < COLONY_PLAN['pilot']['bands']['invalid'] or not pc['plainLoss'] > 0:
        print(f"pilot: invalid {pc['invalid']:.3f}, plain board's honest loss {pc['plainLoss']:.3f}: stop and ask")
        return None
    rule = costRule1(pc['costPerCall'])
    print(f"cost rule: {rule['verdict']} (projected ${rule['projected']:.2f})")
    return rule['cells']


if not LIVE or STAGE not in ('stage1', 'stage2'):
    print('skipped: needs EXPLRL_LLM_LIVE=1 and EXPLRL_COLONY_STAGE=stage1 (after the pilot\'s verdict is recorded)')
else:
    assert COLONY_PLAN['frozen'], 'COLONY_PLAN must be frozen'
    cells = stage1Cells()
    if cells is None:
        print('Stage 1 not run: stop and ask')
    else:
        rows1, S1, _ = runColony('stage1', [('colony1', cells)], LIVE_SPEC)
        print(open(demo_artifact_dir('stage1', COLONY) / 'report.md').read()[:4000])
        print('go/no-go:', goNoGo(rows1))

## Stage 2 (`--cell-timeout 0 --flags llm`, live, about $77-108; only on `go` and a raised cap)

In [ ]:
#| llm
if not LIVE or STAGE != 'stage2':
    print('skipped: needs EXPLRL_LLM_LIVE=1 and EXPLRL_COLONY_STAGE=stage2')
elif not (COLONY2_PLAN['frozen'] and COLONY2_PLAN['capRaised'] and COLONY2_PLAN['arm']):
    print('skipped: Stage 2 needs a go verdict, the user-raised cap and COLONY2_PLAN frozen with the chosen arm')
else:
    pr = readCSV(demo_artifact_dir('pilot', COLONY) / 'runs.csv')
    rule = costRule2(pilotCheck(pr)['costPerCall'] * COLONY2_PLAN['priceRatio'], COLONY2_PLAN['arm'])
    print(f"Stage 2 cost rule: {rule['verdict']} (projected ${rule['projected']:.2f})")
    rows1 = readCSV(demo_artifact_dir('stage1', COLONY) / 'runs.csv')
    plan2 = dict(COLONY_PLAN, sizes=COLONY_PLAN['sizes'] + COLONY2_PLAN['sizes'], stage2=dict(arm=COLONY2_PLAN['arm'], sizes=COLONY2_PLAN['sizes']))
    rows2, S2, _ = runColony('stage2', [('colony2', rule['cells'])], LIVE_SPEC, plan=plan2, prior=rows1)
    print('trend (H18-5, confirmatory):', S2.get('trend'))
    print(open(demo_artifact_dir('stage2', COLONY) / 'report.md').read()[:4000])

## Guards at 32 agents (`--cell-timeout 0 --flags llm`, live, `EXPLRL_COLONY_STAGE=stage32`; a user-approved deviation)

The plain board and guards at N = 32, clean and with independent liars, mistaken members and colluding liars (64 cells, about 67,000 calls); `COLONY32_PLAN` frozen first; the cap raised to $200 for this run. Stage 1's rows give the 8 and 16 agent points of the trend.

In [ ]:
#| llm
def run32():
    spec = dict(LIVE_SPEC, cap=COLONY32_PLAN['cap'])
    rows1 = readCSV(demo_artifact_dir('stage1', COLONY) / 'runs.csv')
    return runColony('stage32', [('colony32', stage32Grid())], spec, plan=COLONY32_PLAN, prior=rows1,
                     analyse=lambda rows, prior: (stage32Summaries(rows, prior, nboot=COLONY32_PLAN['nboot']), writeStage32))


if not LIVE or STAGE != 'stage32':
    print('skipped: needs EXPLRL_LLM_LIVE=1 and EXPLRL_COLONY_STAGE=stage32')
else:
    assert COLONY32_PLAN['frozen'], 'freeze and commit COLONY32_PLAN before the run'
    rows32, S32, _ = run32()
    print(open(demo_artifact_dir('stage32', COLONY) / 'report.md').read())

## Stage 3: adaptive liars and unassigned guards (`--flags llm`, live; `EXPLRL_COLONY_STAGE=stage3pilot`, then `stage3`)

The pilot (seed 1890: `plain|adaptive`, `colonyVolunteer|independent`) checks that Claude liars actually lie (at least 10% of their turns, at most 20% refusals), invalid replies under 5%, and the cost rule (Stage 3 at most $30). Stage 3 (40 cells at N = 16) runs only if both pass; the six reused Stage 1 cells are replayed from the cache with Stage 3's columns ($0, their Stage 1 columns checked unchanged). `COLONY3_PLAN` must be frozen first.

In [ ]:
#| llm
def writePilot3(path, S, meta):
    open(path, 'w').write(f"# Stage 3 pilot\n\ncompliance: {S['gate']}\n\ncost rule: {S['cost']}\n")


def pilot3(rows, prior=()):
    calls = sum(r['calls'] for r in rows) or 1
    return dict(gate=complianceGate(rows), cost=costRule3(sum(r['cost'] for r in rows) / calls)), writePilot3


def stage3Ready():
    "The pilot's verdicts from its saved rows (None if it has not run)."
    pr = demo_artifact_dir('stage3pilot', COLONY) / 'runs.csv'
    return pilot3(readCSV(pr))[0] if pr.exists() else None


def run3():
    rows1 = readCSV(demo_artifact_dir('stage1', COLONY) / 'runs.csv')
    reused = reusedRows(lambda cfg: cacheOnlyClient(cfg, LLM / 'cache'), rows1)
    writeCSV(reused, demo_artifact_dir('stage3', COLONY) / 'reused.csv')
    return runColony('stage3', [('colony3', stage3Grid())], dict(LIVE_SPEC, cap=COLONY3_PLAN['cap']), plan=COLONY3_PLAN, prior=reused,
                     run=runStage3Cell, analyse=lambda rows, prior: (stage3Summaries(rows, prior, nboot=COLONY3_PLAN['nboot']), writeStage3))


if not LIVE or STAGE not in ('stage3pilot', 'stage3'):
    print('skipped: needs EXPLRL_LLM_LIVE=1 and EXPLRL_COLONY_STAGE=stage3pilot or stage3')
else:
    assert COLONY3_PLAN['frozen'], 'freeze and commit COLONY3_PLAN before the pilot'
    if STAGE == 'stage3pilot':
        _, P3, _ = runColony('stage3pilot', [('colony3Pilot', stage3PilotGrid())], dict(LIVE_SPEC, cap=COLONY3_PLAN['cap']),
                             plan=COLONY3_PLAN, run=runStage3Cell, analyse=pilot3)
        print('compliance gate:', P3['gate'], '\ncost rule:', P3['cost'])
    else:
        v = stage3Ready()
        if v is None or not v['gate']['ok'] or v['cost']['verdict'] != 'run':
            print('Stage 3 not run: the pilot has not passed both gates (stop and ask):', v)
        else:
            rows3, S3, _ = run3()
            print(open(demo_artifact_dir('stage3', COLONY) / 'report.md').read())

## Guards at 64 agents (`--flags llm`, live; `EXPLRL_COLONY_STAGE=stage64pilot`, then `stage64`; a second user-approved deviation)

Lean: the plain board and guards with independent liars at N = 64 (16 boards, about 30,700 calls), no clean boards. The pilot (one guard board, seed 1890) measures the cost per call at 64 agents and checks invalid replies (under 5%); Stage 64 runs only if the pilot passes and the cost rule projects at most $35. Stage 1's and the 32-agent run's rows give the 8-32 points of the trend. `COLONY64_PLAN` must be frozen first.

In [ ]:
#| llm
def pilot64(rows, prior=()):
    return pilotCheck64(rows), lambda path, S, meta: open(path, 'w').write(f"# Stage 64 pilot\n\n{S}\n")


def prior64():
    return readCSV(demo_artifact_dir('stage1', COLONY) / 'runs.csv') + readCSV(demo_artifact_dir('stage32', COLONY) / 'runs.csv')


def run64():
    return runColony('stage64', [('colony64', stage64Grid())], dict(LIVE_SPEC, cap=COLONY64_PLAN['cap']), plan=COLONY64_PLAN,
                     prior=prior64(), analyse=lambda rows, prior: (stage64Summaries(rows, prior, nboot=COLONY64_PLAN['nboot']), writeStage64))


def runPilot64():
    return runColony('stage64pilot', [('colony64Pilot', stage64PilotGrid())], dict(LIVE_SPEC, cap=COLONY64_PLAN['cap']),
                     plan=COLONY64_PLAN, analyse=pilot64)


if not LIVE or STAGE not in ('stage64pilot', 'stage64'):
    print('skipped: needs EXPLRL_LLM_LIVE=1 and EXPLRL_COLONY_STAGE=stage64pilot or stage64')
else:
    assert COLONY64_PLAN['frozen'], 'freeze and commit COLONY64_PLAN before the pilot'
    if STAGE == 'stage64pilot':
        _, P64, _ = runPilot64()
        print('Stage 64 pilot:', P64)
    else:
        pr = demo_artifact_dir('stage64pilot', COLONY) / 'runs.csv'
        v = pilotCheck64(readCSV(pr)) if pr.exists() else None
        if v is None or not v['ok'] or v['verdict'] != 'run':
            print('Stage 64 not run: the pilot has not passed (stop and ask):', v)
        else:
            rows64, S64, _ = run64()
            print(open(demo_artifact_dir('stage64', COLONY) / 'report.md').read())

## Stage 4 regression gate (`--flags llm`, without `EXPLRL_LLM_LIVE`; $0)

Every earlier live cell (the pilot, Stage 1, 32 agents, Stage 3 and its pilot, 64 agents and its pilot) replayed through the cache-only client: a request the cache does not hold raises `CacheMiss`, so nothing can be spent; every column of each replayed row must equal its saved row (AC-13). Run after any change to the board, prompts or runners, before the live-flag cached reruns.

In [ ]:
#| llm
from explRL.colony3 import stage3Grid as _s3
from explRL.colony32 import stage32Grid as _s32

if LIVE:
    print('skipped: the regression gate runs without EXPLRL_LLM_LIVE (cache-only)')
else:
    for mode, cells in (('pilot', colonyPilotGrid()), ('stage1', stage1Grid()), ('stage32', _s32()), ('stage3pilot', stage3PilotGrid()),
                        ('stage3', _s3()), ('stage64pilot', stage64PilotGrid()), ('stage64', stage64Grid())):
        f = demo_artifact_dir(mode, COLONY) / 'runs.csv'
        if not f.exists():
            print(f'{mode}: skipped, no live run')
            continue
        chk = replayCheck(cells, readCSV(f), LIVE_SPEC['cache'])
        assert not chk['mismatches'], (mode, chk['mismatches'][:3])
        print(f"{mode}: {chk['checked']} cells replayed from the cache, every saved column equal")

## Stage 4: guards that scale (`--flags llm`, live; `EXPLRL_COLONY_STAGE=stage4pilot`, then `stage4`)

Guard copies (`guardDeal`), failure flags (`failFlag`) and squads of 8 (`squads`) on top of assigned guards, plus two combined arms, at 16 and 64 agents (80 boards, 87,040 calls, about $48); the 56 reference cells (Stage 1 and the 64-agent run) are replayed from the cache ($0) with the new columns and saved as `reused.csv`. The pilot (three boards, seed 1890) checks invalid replies, guard coverage and contest rate, flags on true values and the squad prompt length, and projects the cost (run if at most $60). `STAGE4_PLAN` must be frozen, with the committed forecast's hash, first ([stage4.md](../specs/multiagent/phase18/stage4.md)).

In [ ]:
#| llm
import hashlib


def pilot4(rows, prior=()):
    ref = ref8Tokens(readCSV(demo_artifact_dir('stage1', COLONY) / 'runs.csv'))
    return pilotCheck4(rows, ref), lambda path, S, meta: open(path, 'w').write(f"# Stage 4 pilot\n\n{S}\n")


def reused4():
    d = demo_artifact_dir('stage4', COLONY)
    d.mkdir(parents=True, exist_ok=True)
    prior = readCSV(demo_artifact_dir('stage1', COLONY) / 'runs.csv') + readCSV(demo_artifact_dir('stage64', COLONY) / 'runs.csv')
    rows = reusedRows4(lambda cfg: cacheOnlyClient(cfg, LIVE_SPEC['cache']), prior, reusedCells4())
    writeCSV(rows, d / 'reused.csv')
    return rows


def run4():
    return runColony('stage4', [('colony4', stage4Grid())], dict(LIVE_SPEC, cap=STAGE4_PLAN['cap']), plan=STAGE4_PLAN,
                     prior=reused4(), run=runStage4Cell,
                     analyse=lambda rows, prior: (stage4Summaries(rows, prior, nboot=STAGE4_PLAN['nboot']), writeStage4))


def runPilot4():
    return runColony('stage4pilot', [('colony4Pilot', stage4PilotGrid())], dict(LIVE_SPEC, cap=STAGE4_PLAN['cap']),
                     plan=STAGE4_PLAN, run=runStage4Cell, analyse=pilot4)


BATCHES4 = {'stage4b1': COLONY_SEEDS[:4], 'stage4b2': COLONY_SEEDS[4:]}   # seed batches of the full grid; every batch runs


def batch4(rows, prior=()):
    "A batch is not analysed: its boards, invalid share and cost only (the frozen analysis runs on the full grid)."
    calls = sum(r['calls'] for r in rows) or 1
    S = dict(boards=len(rows), invalid=sum(r['invalid'] for r in rows) / calls, cost=sum(r['cost'] for r in rows))
    return S, lambda path, S, meta: open(path, 'w').write(f"# Stage 4 batch (not analysed)\n\n{S}\n")


def runBatch4(mode):
    return runColony(mode, [('colony4', stage4Grid(BATCHES4[mode]))], dict(LIVE_SPEC, cap=STAGE4_PLAN['cap']), plan=STAGE4_PLAN,
                     run=runStage4Cell, analyse=batch4)


def forecast4Ok():
    return FORECAST4.exists() and hashlib.sha256(FORECAST4.read_bytes()).hexdigest() == STAGE4_PLAN['forecast']['sha256']


if not LIVE or STAGE not in ('stage4pilot', 'stage4b1', 'stage4b2', 'stage4'):
    print('skipped: needs EXPLRL_LLM_LIVE=1 and EXPLRL_COLONY_STAGE=stage4pilot, stage4b1, stage4b2 or stage4')
else:
    assert STAGE4_PLAN['frozen'] and forecast4Ok(), 'commit the Stage 4 forecast, then freeze and commit STAGE4_PLAN with its hash'
    if STAGE == 'stage4pilot':
        _, P4, _ = runPilot4()
        print('Stage 4 pilot:', P4)
    else:
        pr = demo_artifact_dir('stage4pilot', COLONY) / 'runs.csv'
        v = pilot4(readCSV(pr))[0] if pr.exists() else None
        b1 = demo_artifact_dir('stage4b1', COLONY) / 'runs.csv'
        inv1 = batch4(readCSV(b1))[0]['invalid'] if b1.exists() else None
        if v is None or not v['ok']:
            print('Stage 4 not run: the pilot has not passed (stop and ask):', v)
        elif STAGE == 'stage4b2' and (inv1 is None or inv1 >= STAGE4_PLAN['pilot']['maxInvalid']):
            print('batch 2 not run: batch 1 is missing or its invalid share reached 5% (stop and ask):', inv1)
        elif STAGE in BATCHES4:
            _, SB, _ = runBatch4(STAGE)
            print(f'{STAGE}:', SB)
        else:
            run4()
            print(open(demo_artifact_dir('stage4', COLONY) / 'report.md').read())

## Cached reruns (`--flags llm`, after the live runs)

Every reply from the cache: `runs.csv` and `turns.jsonl` byte-identical, nothing spent (AC-10).

In [ ]:
#| llm
if not LIVE:
    print('skipped: set EXPLRL_LLM_LIVE=1')
else:
    def stage2Cells():
        if not (COLONY2_PLAN['frozen'] and COLONY2_PLAN['arm']):
            return None
        pr = readCSV(demo_artifact_dir('pilot', COLONY) / 'runs.csv')
        return costRule2(pilotCheck(pr)['costPerCall'] * COLONY2_PLAN['priceRatio'], COLONY2_PLAN['arm'])['cells']

    for mode3, again in (('stage64pilot', runPilot64), ('stage64', run64), ('stage3pilot', lambda: runColony('stage3pilot', [('colony3Pilot', stage3PilotGrid())], dict(LIVE_SPEC, cap=COLONY3_PLAN['cap']),
                                                       plan=COLONY3_PLAN, run=runStage3Cell, analyse=pilot3)), ('stage3', run3)):
        f3 = [demo_artifact_dir(mode3, COLONY) / f for f in ('runs.csv', 'turns.jsonl')]
        if f3[0].exists() and COLONY3_PLAN['frozen'] and (not mode3.startswith('stage64') or COLONY64_PLAN['frozen']):
            before = [p.read_bytes() for p in f3]
            again()
            assert [p.read_bytes() for p in f3] == before
            print(f'{mode3}: cached rerun byte-identical')
    for mode4, again in (('stage4pilot', runPilot4), ('stage4b1', lambda: runBatch4('stage4b1')),
                         ('stage4b2', lambda: runBatch4('stage4b2')), ('stage4', run4)):
        f4 = [demo_artifact_dir(mode4, COLONY) / f for f in ('runs.csv', 'turns.jsonl')]
        if f4[0].exists() and STAGE4_PLAN['frozen']:
            before = [p.read_bytes() for p in f4]
            again()
            assert [p.read_bytes() for p in f4] == before
            print(f'{mode4}: cached rerun byte-identical')
    files32 = [demo_artifact_dir('stage32', COLONY) / f for f in ('runs.csv', 'turns.jsonl')]
    if files32[0].exists() and COLONY32_PLAN['frozen']:
        before = [p.read_bytes() for p in files32]
        run32()
        assert [p.read_bytes() for p in files32] == before
        print('stage32: cached rerun byte-identical')
    for mode, grid in (('pilot', lambda: colonyPilotGrid()), ('stage1', stage1Cells), ('stage2', stage2Cells)):
        files = [demo_artifact_dir(mode, COLONY) / f for f in ('runs.csv', 'turns.jsonl')]
        if not files[0].exists() or (cells := grid()) is None:
            print(f'{mode}: skipped, no live run')
            continue
        before, spent = [p.read_bytes() for p in files], Budget(CAP, LIVE_SPEC['ledger']).spent()
        study = dict(pilot='colonyPilot', stage1='colony1', stage2='colony2')[mode]
        plan = COLONY_PLAN if mode != 'stage2' else dict(COLONY_PLAN, sizes=COLONY_PLAN['sizes'] + COLONY2_PLAN['sizes'],
                                                          stage2=dict(arm=COLONY2_PLAN['arm'], sizes=COLONY2_PLAN['sizes']))
        prior = readCSV(demo_artifact_dir('stage1', COLONY) / 'runs.csv') if mode == 'stage2' else ()
        runColony(mode, [(study, cells)], LIVE_SPEC, plan=plan, prior=prior)
        assert [p.read_bytes() for p in files] == before and Budget(CAP, LIVE_SPEC['ledger']).spent() == spent, mode
        print(f'{mode}: cached rerun byte-identical, free')